In [3]:
import tensorflow.compat.v1 as tf
import numpy as np
tf.disable_v2_behavior() 

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 归一化及重塑形状
x_train = x_train.reshape(-1, 784) / 255.0
x_test = x_test.reshape(-1, 784) / 255.0
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 定义数据加载器类
class MNIST_Loader:
    def __init__(self, x, y):
        self.x, self.y, self.idx = x, y, 0
    def next_batch(self, batch_size):
        if self.idx + batch_size > len(self.x):
            self.idx = 0 
        batch_x = self.x[self.idx : self.idx + batch_size]
        batch_y = self.y[self.idx : self.idx + batch_size]
        self.idx += batch_size
        return batch_x, batch_y

mnist_train = MNIST_Loader(x_train, y_train)

learning_rate = 1e-4
keep_prob_rate = 0.7 
max_epoch = 2000

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

# 卷积函数
def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

# 池化函数
def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

def compute_accuracy(v_xs, v_ys, sess, prediction_op, xs_ph, keep_prob_ph):
    y_pre = sess.run(prediction_op, feed_dict={xs_ph: v_xs, keep_prob_ph: 1.0})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    return sess.run(accuracy)

# 定义占位符
xs = tf.placeholder(tf.float32, [None, 784])
ys = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

# 第一层卷积
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 第二层卷积
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层 1
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2 (输出层)
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

# 损失与优化
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction + 1e-10), reduction_indices=[1]))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    print("开始训练...")
    for i in range(max_epoch):
        batch_xs, batch_ys = mnist_train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        
        if i % 100 == 0:
            acc = compute_accuracy(x_test[:1000], y_test[:1000], sess, prediction, xs, keep_prob)
            print(f"步数 {i:4d}, 测试集准确率: {acc:.4f}")

    final_acc = compute_accuracy(x_test, y_test, sess, prediction, xs, keep_prob)
    print(f"训练结束，全量测试集最终准确率: {final_acc:.4f}")

开始训练...
步数    0, 测试集准确率: 0.2530
步数  100, 测试集准确率: 0.8630
步数  200, 测试集准确率: 0.9120
步数  300, 测试集准确率: 0.9380
步数  400, 测试集准确率: 0.9430
步数  500, 测试集准确率: 0.9570
步数  600, 测试集准确率: 0.9600
步数  700, 测试集准确率: 0.9670
步数  800, 测试集准确率: 0.9730
步数  900, 测试集准确率: 0.9730
步数 1000, 测试集准确率: 0.9730
步数 1100, 测试集准确率: 0.9810
步数 1200, 测试集准确率: 0.9760
步数 1300, 测试集准确率: 0.9750
步数 1400, 测试集准确率: 0.9810
步数 1500, 测试集准确率: 0.9750
步数 1600, 测试集准确率: 0.9850
步数 1700, 测试集准确率: 0.9870
步数 1800, 测试集准确率: 0.9830
步数 1900, 测试集准确率: 0.9860
训练结束，全量测试集最终准确率: 0.9816
